<a href="https://colab.research.google.com/github/GDVevo/ml_uni/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9612.2_%D0%93%D0%B5%D0%BE%D0%BC%D0%B0%D1%80%D0%BA%D0%B5%D1%82%D0%B8%D0%BD%D0%B3%D0%BE%D0%B2%D0%BE%D0%B5_%D0%B8%D1%81%D1%81%D0%BB%D0%B5%D0%B4%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D1%82%D0%B5%D1%80%D1%80%D0%B8%D1%82%D0%BE%D1%80%D0%B8%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа. Геомаркетинговое исследование территорий с применением методов машинного обучения**


## **Цель работы**


Овладеть методами пространственного анализа и машинного обучения для решения практической задачи определения оптимальных локаций размещения коммерческих объектов.

## **Введение**


В современном бизнесе местоположение коммерческого объекта играет ключевую роль в его успешности. Геомаркетинговый анализ позволяет объективно оценить привлекательность различных локаций, опираясь на количественные показатели и алгоритмы машинного обучения.

В рамках данной работы вы примените полный цикл пространственного анализа, включая сбор данных из открытых источников, обработку и агрегацию пространственной информации, обучение моделей машинного обучения и визуализацию результатов.

## **Задание**


Провести геомаркетинговое исследование для выбора оптимальных локаций размещения новых точек определенного типа бизнеса на территории выбранного города.

## **Порядок выполнения работы**

### **Часть 1. Подготовка данных и определение задачи**



1. **Индивидуальный выбор территории и типа бизнеса**:
   - Выберите город/район для проведения анализа (например, центральная часть Москвы, Санкт-Петербурга или любого другого крупного города)

**Ответ**: выберем город Подольск, просто потому что :)


   - Определите тип бизнеса для анализа (аптеки, продуктовые магазины, пункты выдачи заказов, рестораны определенной кухни и т.д.)

   **Ответ**: были выбраны ПВЗ Ozon (стоит сразу отметить, что у самой компании уже существует карта для подбора местоположения открытия ПВЗ. возможно, будет интересно сравнить результаты)

   - Обоснуйте свой выбор: почему данная территория и тип бизнеса интересны для анализа?

   **Ответ**: ПВЗ Ozon интересны для анализа того, например, где лучше всего будет открыть ещё один ПВЗ, как часто пользуются (если есть такая информация) уже имеющимися ПВЗ.

   Подольск может быть интересен для сотрудников Ozon из-за своей населенности, мер развития

2. **Определение ключевых факторов успешности**:
   - Самостоятельно сформулируйте не менее 5 факторов, которые могут влиять на успешность выбранного типа бизнеса

   **Ответ**:
- **Локация**. Наиболее важный пункт, насколько удобно добраться до ПВЗ, какой размер территории он может покрыть
- **Конкуренция**. Плотность пунктов в жилом массиве, необходимо выбирать локации, где поблизости нет других ПВЗ того же или конкурирующих маркетплейсов.
- **Трафик**. Находятся ли поблизости остановки, станции метро; крупные здания по типу школ, садов, офисов и т.д.
- **Площадь**. Большая площадь помещения может быть более благоприятной для ПВЗ, так как позволяет создавать более крупный склад для хранения товаров
- **Наличие поблизости парковок**. Возможно, некоторым людям всегда удобнее будет приезжать к ПВЗ. Также это позволяет людям забирать более крупные заказы на машине

   - Для каждого фактора определите, какими данными из OpenStreetMap его можно количественно описать

**Ответ**:

- *Локация*:

      building=apartments, building=residential
      landuse=residential
      addr:housenumber=* (подсчет количества адресов в полигоне)

- *Конкуренция*:    
  - Прямые конкуренты (ПВЗ):
        
        shop=outpost (универсальный тег для ПВЗ)
        amenity=parcel_locker (постаматы)
        brand=Ozon / brand=Wildberries / brand=Yandex Market (если бренд указан в тегах, что бывает не всегда)
        operator=* (часто оператор указан вместо бренда)
  - Смежные конкуренты (почта/логистика):

        amenity=post_office
        shop=courier

- *Трафик*:

      shop=supermarket, shop=mall, amenity=marketplace
      public_transport=stop_position (остановки), railway=station (метро/электрички)
      amenity=school, amenity=kindergarten, office=*

- *Площадь*:

  - Явные указания площади (редко, но бывает):

        floor_area=* (общая площадь в м²)
        rooms=* (количество комнат; для ПВЗ желательно ≥ 2: зал выдачи + склад)

  - Характеристики, влияющие на полезную площадь:

        indoor=room + room=storage (наличие выделенного склада внутри)
        ceiling:height=* (высота потолков; важно для стеллажей)
        door:width=* (ширина входной двери; критично для заноса крупногабарита)

- *Наличие поблизости парковок*:

  Ключевые теги в радиусе 50–100 м от входа:

        amenity=parking
        parking=street_side (парковка вдоль дороги)
        parking=lane (парковочная полоса)
        access=customers или access=public (важно! частные парковки private не подходят)
        highway=living_street (дворовая территория, часто есть стихийная парковка)
        fee=no (бесплатная парковка предпочтительнее)

   - Составьте таблицу соответствия между факторами и тегами OpenStreetMap

|Фактор|Тэг|
|-|-|
|Локация|building=apartments, building=residential<br>landuse=residential<br>addr:housenumber=* (подсчет количества адресов в полигоне)|
|Конкуренция|shop=outpost (универсальный тег для ПВЗ)<br>amenity=parcel_locker (постаматы)<br>brand=Ozon / brand=Wildberries / brand=Yandex Market (если бренд указан в тегах)<br>operator=* (часто оператор указан вместо бренда)|
|Трафик|shop=supermarket, shop=mall, amenity=marketplace<br>public_transport=stop_position (остановки), railway=station (метро/электрички)<br>amenity=school, amenity=kindergarten, office=*|
|Площадь|floor_area=* (общая площадь в м²)<br>rooms=* (количество комнат; для ПВЗ желательно ≥ 2: зал выдачи + склад)<br>indoor=room + room=storage (наличие выделенного склада внутри)<br>ceiling:height=* (высота потолков; важно для стеллажей)<br>door:width=* (ширина входной двери; критично для заноса крупногабарита)|
|Наличие парковок|amenity=parking<br>parking=street_side (парковка вдоль дороги)<br>parking=lane (парковочная полоса)<br>access=customers или access=public (важно! частные парковки private не подходят)<br>highway=living_street (дворовая территория, часто есть стихийная парковка)<br>fee=no (бесплатная парковка предпочтительнее)|

3. **Сбор исходных данных**:
   - Настройте необходимые библиотеки из теоретического материала

In [1]:
!pip install osmnx geopandas leafmap mapclassify h3pandas h3~=3.0
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import leafmap
import osmnx as ox
from shapely.geometry import box, Polygon
from sklearn.preprocessing import MinMaxScaler
import h3
import h3pandas

   - Определите необходимую область интереса (ROI) с помощью интерактивной карты


In [2]:
m = leafmap.Map(draw_control=True, basemap='CartoDB.Positron')
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [3]:
# bbox = m.user_roi_bounds()
# print(m.center)
# print(bbox)

   - Загрузите данные о существующих объектах вашего типа бизнеса и объектах инфраструктуры, связанных с выделенными вами факторами

In [4]:
ozon_gdf = ox.features_from_place("Подольск", tags={'name':'Ozon', 'brand':'Ozon'})
ozon_gdf

geometry brand brand:wikidata  \
element id                                                            
node    9076606402   POINT (37.52601 55.35784)  Ozon       Q2365235   
        9076606409    POINT (37.52271 55.4289)  Ozon       Q2365235   
        9076607465   POINT (37.54375 55.42832)  Ozon       Q2365235   
        9076607690   POINT (37.48973 55.41928)  Ozon       Q2365235   
        9076607748   POINT (37.56635 55.43924)  Ozon       Q2365235   
...                                        ...   ...            ...   
        12317955686  POINT (37.54717 55.43299)  Ozon       Q2365235   
        12317968511  POINT (37.54465 55.38238)  Ozon       Q2365235   
        12558494604  POINT (37.52876 55.41438)  Ozon       Q2365235   
        13367874811  POINT (37.53276 55.37018)   NaN            NaN   
        13367874829  POINT (37.54206 55.37328)   NaN            NaN   

                    brand:wikipedia                  contact:facebook  \
element id                                                              
node    9076606402       ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        9076606409       ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        9076607465       ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        9076607690       ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        9076607748       ru:Ozon.ru  https://www.facebook.com/ozon.ru   
...                             ...                               ...   
        12317955686      ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        12317968511      ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        12558494604      ru:Ozon.ru  https://www.facebook.com/ozon.ru   
        13367874811             NaN                               NaN   
        13367874829             NaN                               NaN   

                                    contact:instagram   contact:phone  \
element id                                                              
node    9076606402   https://www.instagram.com/ozonru  +7 495 2321000   
        9076606409   https://www.instagram.com/ozonru  +7 495 2321000   
        9076607465   https://www.instagram.com/ozonru  +7 495 2321000   
        9076607690   https://www.instagram.com/ozonru  +7 495 2321000   
        9076607748   https://www.instagram.com/ozonru  +7 495 2321000   
...                                               ...             ...   
        12317955686  https://www.instagram.com/ozonru  +7 495 2321000   
        12317968511  https://www.instagram.com/ozonru  +7 495 2321000   
        12558494604  https://www.instagram.com/ozonru  +7 495 2321000   
        13367874811                               NaN  +7 495 2321000   
        13367874829                               NaN  +7 495 2321000   

                                 contact:twitter           contact:vk  \
element id                                                              
node    9076606402   https://twitter.com/Ozon_ru  https://vk.com/ozon   
        9076606409   https://twitter.com/Ozon_ru  https://vk.com/ozon   
        9076607465   https://twitter.com/Ozon_ru  https://vk.com/ozon   
        9076607690   https://twitter.com/Ozon_ru  https://vk.com/ozon   
        9076607748   https://twitter.com/Ozon_ru  https://vk.com/ozon   
...                                          ...                  ...   
        12317955686  https://twitter.com/Ozon_ru  https://vk.com/ozon   
        12317968511  https://twitter.com/Ozon_ru  https://vk.com/ozon   
        12558494604  https://twitter.com/Ozon_ru  https://vk.com/ozon   
        13367874811                          NaN                  NaN   
        13367874829                          NaN                  NaN   

                          contact:website  name      opening_hours     shop  \
element id                                                                    
node    9076606402   https://www.ozon.ru/  Ozon  Mo-Su 10:00-22:00  outpost   
        9076606409    https

In [6]:
bbox = m.user_roi_bounds()
if bbox:
    print(f"Выбранный bounding box: {bbox}")
else:
    bbox = [37.444, 55.2955, 37.6569, 55.496]
    print(f"Область не выбрана. Используем значение по умолчанию: {bbox}")
m2 = leafmap.Map(center=[(bbox[1]+bbox[3])/2, (bbox[0]+bbox[2])/2], zoom=11, basemap='CartoDB.Positron')
m2.add_gdf(ozon_gdf, layer_name='OZON Location', info_mode='on_click')

m2

Выбранный bounding box: [37.4332, 55.3041, 37.6542, 55.4897]


Map(center=[55.3969, 37.5437], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

### **Часть 2. Пространственное агрегирование и создание признаков**


1. **Создание гексагональной сетки**:
   - Самостоятельно определите оптимальную детализацию сетки H3 для вашего анализа

In [9]:
res = 9
crs_proj = "EPSG:3857"

   - Покройте территорию гексагональной сеткой выбранного разрешения

In [10]:
# 1. Получаем полигон территории
area_gdf = ox.geocode_to_gdf("Подольск").to_crs("EPSG:4326")
bbox = area_gdf.total_bounds

# 2. Генерируем H3-индексы (универсальный метод, работает со всеми версиями h3)
bbox_poly = box(*bbox)
h3_ids = list(h3.polyfill_geojson(bbox_poly.__geo_interface__, res))

# 3. Создаём геометрии гексагонов из индексов
h3_geoms = [Polygon(h3.h3_to_geo_boundary(idx, geo_json=True)) for idx in h3_ids]

# 4. Собираем GeoDataFrame
h3_gdf = gpd.GeoDataFrame({'h3_index': h3_ids, 'geometry': h3_geoms}, crs="EPSG:4326")

# 5. Обрезаем по реальной границе города и переводим в метрическую проекцию
h3_gdf = h3_gdf[h3_gdf.intersects(area_gdf.unary_union)].reset_index(drop=True)
h3_gdf = h3_gdf.to_crs(crs_proj)

# 1. Переводим сетку в EPSG:4326 (широта/долгота), так как leafmap/Folium работают именно с ней
h3_gdf_viz = h3_gdf.to_crs("EPSG:4326")

# 2. Добавляем слой на ранее созданную карту m
m3 = leafmap.Map(center=[(bbox[1]+bbox[3])/2, (bbox[0]+bbox[2])/2], zoom=11, basemap='CartoDB.Positron')

m3.add_gdf(
    h3_gdf_viz,
    layer_name="H3 Сетка (res 9)",
    info_mode="on_hover",
    style={
        "stroke": True,
        "color": "#3388ff",
        "weight": 1.5,
        "fillOpacity": 0.15,
        "fillColor": "#3388ff"
    }
)

m3

Map(center=[np.float64(55.39283315), np.float64(37.5398555)], controls=(ZoomControl(options=['position', 'zoom…

- Обоснуйте выбор разрешения (resolution) сетки с учётом масштаба вашей территории и специфики бизнеса

**Ответ**: выбранное разрешение (9, площадь ячейки=~0.11 км², ребро ~174 м) предоставляет баланс между производительностью и детализацией, затрагивает достаточно оптимальную площадь для данного анализа - она и не слишком маленькая/точная, и не слишком большая/распространенная

2. **Инженерия пространственных признаков**:
   - Определите радиусы для анализа ближнего и среднего окружения (могут отличаться от предложенных в теории в зависимости от специфики вашего бизнеса)

**Ответ**:
- Радиус 400 м (~5 мин. пешком): оценка доступности вблизи жилых домой, для более быстрого доступа к ПВЗ
- Радиус 800 м (~10 мин. пешком): оценка доступности в зонах, где затруднительно где-либо расположить ПВЗ, но всё ещё в относительной близости к жилым домам


Разработайте и реализуйте не менее 10 пространственных признаков, описывающих характеристики каждой ячейки

**Дополнительное задание**: придумайте и реализуйте не менее 2 пространственных признаков, которых нет в теоретическом материале

3. **Анализ полученных признаков**:
   - Рассчитайте базовую статистику по каждому признаку

   - Исследуйте корреляции между признаками

   - Выявите и обработайте выбросы и пропущенные значения, если они имеются

### **Часть 3. Моделирование привлекательности локаций**



1. **Подготовка целевой переменной**:
   - Определите, как будет сформирована целевая переменная для вашей задачи (по умолчанию: наличие объектов выбранного типа в ячейке)


   - Исследуйте распределение целевой переменной и оцените её сбалансированность

   - При необходимости, предложите стратегию работы с несбалансированными данными

2. **Разработка моделей машинного обучения**:
   - Реализуйте и обучите несколько моделей (минимум 2) для предсказания привлекательности локации

   - Проведите оценку важности признаков для каждой модели

   - Сравните модели по метрикам качества и выберите наилучшую

   - **Дополнительное задание**: настройте гиперпараметры модели с помощью поиска по сетке (GridSearchCV) или случайного поиска (RandomSearchCV)

3. **Улучшение модели с помощью кластеризации**:
   - Выполните кластеризацию ячеек по их характеристикам


   - Определите оптимальное число кластеров с помощью метода локтя или силуэта

   - Визуализируйте результаты кластеризации

   - Проверьте, улучшает ли добавление информации о кластерах качество основной модели

### **Часть 4. Расчет потенциала локаций и финальные рекомендации**



1. **Разработка интегрального показателя потенциала**:
   - Самостоятельно определите веса для факторов привлекательности среды и конкуренции

   - Рассчитайте итоговый потенциал для всех ячеек сетки

   - Категоризируйте потенциал для упрощения интерпретации результатов

   - Обоснуйте выбранные веса в контексте вашего бизнеса

2. **Визуализация результатов**:
   - Создайте интерактивную карту с тепловым слоем потенциала

   - Добавьте маркеры существующих объектов вашего типа бизнеса

   - Выделите топ-10 локаций с наивысшим потенциалом

   - Подготовьте отдельную карту фокуса на лучших локациях

3. **Формирование бизнес-рекомендаций**:
   - Составьте список из 5-7 конкретных локаций для размещения новых объектов

   - Для каждой рекомендуемой локации укажите:
     * Точные координаты
     * Значение потенциала
     * Ключевые характеристики локации
     * Преимущества и возможные риски размещения в данной точке

   - Подготовьте общие рекомендации по стратегии территориального развития для выбранного бизнеса

## **Рекомендации по выполнению**


1. Начните с малой территории для тестирования кода и методологии, затем расширяйте анализ.
2. Используйте инкрементальный подход: сначала реализуйте базовый функционал, затем улучшайте его.
3. Регулярно сохраняйте промежуточные результаты работы.
4. При выборе признаков опирайтесь не только на учебный материал, но и на научные статьи по геоанализу и геомаркетингу.
5. Обращайте внимание на особенности территории и уникальные характеристики выбранного бизнеса.
6. Для оценки результатов старайтесь сопоставить их с реальным расположением успешных объектов аналогичного бизнеса.